In [2]:
!pip install tensorflow==2.20.0 opencv-python matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

c:\Users\Riya Mahajan\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [19]:
import kagglehub
path = kagglehub.model_download("google/movenet/tfLite/singlepose-lightning-tflite-float16")
print("Path to model files:", path)

100%|██████████| 4.54M/4.54M [00:03<00:00, 1.31MB/s]

Path to model files: C:\Users\Riya Mahajan\.cache\kagglehub\models\google\movenet\tfLite\singlepose-lightning-tflite-float16\1


In [22]:
import os
model_file = os.path.join(path, "4.tflite")
interpreter = tf.lite.Interpreter(model_path=model_file)
interpreter.allocate_tensors()

In [28]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Cannot open camera")
    exit()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
while True:
    ret, frame = cap.read()

    if not ret or frame is None:
        print("Failed to grab frame")
        break

    try:
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = tf.image.resize_with_pad(img, 192, 192)
        input_image = np.expand_dims(img, axis=0).astype(np.uint8)

        interpreter.set_tensor(input_details[0]['index'], input_image)
        interpreter.invoke()
        keypoints_with_scores = interpreter.get_tensor(output_details[0]['index'])

        draw_connections(frame, keypoints_with_scores, EDGES, 0.4)
        draw_keypoints(frame, keypoints_with_scores, 0.4)

    except Exception as e:
        print("Error during inference:", e)
        break

    cv2.imshow("MoveNet Lightning", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [20]:
import os
print(os.listdir(path))

['4.tflite']


In [25]:
EDGES = {
    (0, 1): 'm',
    (0, 2): 'c',
    (1, 3): 'm',
    (2, 4): 'c',
    (0, 5): 'm',
    (0, 6): 'c',
    (5, 7): 'm',
    (7, 9): 'm',
    (6, 8): 'c',
    (8, 10): 'c',
    (5, 6): 'y',
    (5, 11): 'm',
    (6, 12): 'c',
    (11, 12): 'y',
    (11, 13): 'm',
    (13, 15): 'm',
    (12, 14): 'c',
    (14, 16): 'c'
}

In [26]:
def draw_connections(frame, keypoints, edges, confidence_threshold):
    y, x, c = frame.shape
    shaped = np.squeeze(np.multiply(keypoints, [y,x,1]))
    
    for edge, color in edges.items():
        p1, p2 = edge
        y1, x1, c1 = shaped[p1]
        y2, x2, c2 = shaped[p2]
        
        if (c1 > confidence_threshold) & (c2 > confidence_threshold):      
            cv2.line(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0,0,255), 2)

In [27]:
def draw_keypoints(frame, keypoints, confidence_threshold):
    y, x, c = frame.shape
    shaped = np.squeeze(np.multiply(keypoints, [y,x,1]))
    
    for kp in shaped:
        ky, kx, kp_conf = kp
        if kp_conf > confidence_threshold:
            cv2.circle(frame, (int(kx), int(ky)), 4, (0,255,0), -1) 